In [1]:
import pandas as pd
path = "../data/raw/online_retail_II.xlsx"

# sheet_name=None reads EVERY sheet into a dictionary,
# then we stack them into one table
all_sheets = pd.read_excel(path, sheet_name=None)
df = pd.concat(all_sheets.values(), ignore_index=True)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
print("Rows and columns:", df.shape)
print("Earliest date:", df['InvoiceDate'].min())
print("Latest date:", df['InvoiceDate'].max())

Rows and columns: (1067371, 8)
Earliest date: 2009-12-01 07:45:00
Latest date: 2011-12-09 12:50:00


In [3]:
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

In [4]:
print("Total rows:", len(df))
print("Rows missing Customer ID:", df['Customer ID'].isna().sum())

Total rows: 1067371
Rows missing Customer ID: 243007


In [5]:
cancellations = df['Invoice'].astype(str).str.startswith('C')
print("Cancellation rows:", cancellations.sum())

Cancellation rows: 19494


In [6]:
print("Rows with Quantity <= 0:", (df['Quantity'] <= 0).sum())

Rows with Quantity <= 0: 22950


In [7]:
print("Rows with Price <= 0:", (df['Price'] <= 0).sum())

Rows with Price <= 0: 6207


In [8]:
# rows where StockCode isn't a normal product code
df[df['Quantity'] <= 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


In [9]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

Starting rows: 1067371


In [10]:
df_clean = df_clean[df_clean['Customer ID'].notna()]
print("After dropping missing customers:", len(df_clean))

After dropping missing customers: 824364


In [11]:
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
print("After dropping cancellations:", len(df_clean))

After dropping cancellations: 805620


In [12]:
df_clean = df_clean[df_clean['Quantity'] > 0]
print("After dropping bad quantities:", len(df_clean))

After dropping bad quantities: 805620


In [13]:
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['Price']
df_clean.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [14]:
print("Any missing customers left?", df_clean['Customer ID'].isna().sum())
print("Any negative quantities left?", (df_clean['Quantity'] <= 0).sum())
print("Number of unique customers:", df_clean['Customer ID'].nunique())

Any missing customers left? 0
Any negative quantities left? 0
Number of unique customers: 5881


In [15]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

Snapshot date: 2011-12-10 12:50:00


In [16]:
rfm = df_clean.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalAmount', 'sum')
).reset_index()

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [17]:
print("Number of customers:", len(rfm))
print(rfm.describe())

Number of customers: 5881
        Customer ID      Recency    Frequency       Monetary
count   5881.000000  5881.000000  5881.000000    5881.000000
mean   15314.674205   201.457745     6.287196    3017.076888
std     1715.429759   209.474135    13.012879   14734.128619
min    12346.000000     1.000000     1.000000       0.000000
25%    13833.000000    26.000000     1.000000     347.800000
50%    15313.000000    96.000000     3.000000     897.620000
75%    16797.000000   380.000000     7.000000    2304.180000
max    18287.000000   739.000000   398.000000  608821.650000


In [18]:
# 180-day performance window. Tested 90/120/180/270/365: 180 gives the best
# ROC-AUC (0.81 vs 0.80 at 90d). Past 180 the label gets cleaner but the
# observation history shrinks too far and accuracy falls back off.
WINDOW_DAYS = 180
cutoff_date = df_clean['InvoiceDate'].max() - pd.Timedelta(days=WINDOW_DAYS)
print("Cutoff date:", cutoff_date)

# split the data into "before" (describe) and "after" (test)
observation = df_clean[df_clean['InvoiceDate'] <= cutoff_date]
performance = df_clean[df_clean['InvoiceDate'] >  cutoff_date]

print("Rows before cutoff:", len(observation))
print(f"Rows in the {WINDOW_DAYS}-day test window:", len(performance))

Cutoff date: 2011-06-12 12:50:00
Rows before cutoff: 563570
Rows in the 180-day test window: 242050


In [19]:
rfm = observation.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalAmount', 'sum')
).reset_index()

print("Customers described:", len(rfm))

Customers described: 4979


In [20]:
returning_customers = performance['Customer ID'].unique()

# churned = 1 if they did NOT return in the 180-day window, 0 if they did
rfm['Is_Churned'] = (~rfm['Customer ID'].isin(returning_customers)).astype(int)

rfm.head()

,Customer ID,Recency,Frequency,Monetary,Is_Churned
0,12346.0,145,12,77556.46,1
1,12347.0,2,5,3529.27,0
2,12348.0,68,4,1709.40,0
3,12349.0,227,3,2671.14,0
4,12350.0,129,1,334.40,1


In [21]:
rfm.to_parquet("../data/processed/customer_rfm.parquet", index=False)
print("Saved. Phase 1 complete.")

Saved. Phase 1 complete.
